# Crawling review

- 순서: 학과 목록 → 수집 시작 주소 → URL 수집 → 본문 추출 → 페이지 유형 검증 → 수동 검토 → 최종 데이터셋
- 각 단계의 `RUN_*` 값을 `True`로 바꾼 셀만 실행합니다. 기본값은 모두 `False`입니다.
- 네트워크 수집과 API 호출은 별도 허용값이 필요합니다. 수동 검토표는 사람이 확인한 뒤 반영합니다.
- `ESG_RESEARCH_ROOT`는 `comp`의 상위 ESG 연구 폴더를 가리킵니다. 실행 결과에 원문이 표시될 수 있으므로 공개 전 출력은 지웁니다.
- 이 노트북은 현재 공개 코드를 단계별로 확인하는 용도이며, 당시 작업 기록을 재현한 것은 아닙니다.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

RESEARCH_ROOT = Path(os.environ.get(
    "ESG_RESEARCH_ROOT",
    str(Path.home() / "Desktop" / "JHI" / "Lab Project" / "ESG"),
)).expanduser()

def find_script_dir():
    here = Path.cwd()
    candidates = [here, here / "code" / "crawling", here / "ESG_comp" / "code" / "crawling", here.parent / "crawling"]
    return next((p for p in candidates if (p / "build_entity_registry.py").is_file()), None)

SCRIPT_DIR = find_script_dir()
assert SCRIPT_DIR is not None, "SCRIPT_DIR를 공개 code/crawling 경로로 지정하세요."
COMP_ROOT = RESEARCH_ROOT / "comp"
CRAWLING_DATA = COMP_ROOT / "0_crawling"
ALLOW_LOCAL_WRITES = False
ALLOW_NETWORK = False
ALLOW_API = False
ALLOW_MANUAL_APPLY = False
CONFIRMED_MANUAL_REVIEW = False
ALLOW_REPLACE_FINAL_DATASET = False
SHOW_PRIVATE_ROWS = False
print({"scripts": str(SCRIPT_DIR), "private_data": str(CRAWLING_DATA)})

In [ ]:
def run_step(script, *args, network=False, api=False, manual_apply=False):
    if not ALLOW_LOCAL_WRITES:
        raise RuntimeError("쓰기 작업 전 ALLOW_LOCAL_WRITES=True로 설정하세요.")
    if network and not ALLOW_NETWORK:
        raise RuntimeError("네트워크 수집 전 ALLOW_NETWORK=True로 설정하세요.")
    if api and not ALLOW_API:
        raise RuntimeError("API 호출 전 ALLOW_API=True로 설정하세요.")
    if manual_apply and not ALLOW_MANUAL_APPLY:
        raise RuntimeError("검토표 확인 후 ALLOW_MANUAL_APPLY=True로 설정하세요.")
    assert COMP_ROOT.is_dir(), f"비공개 comp 폴더가 없습니다: {COMP_ROOT}"
    env = os.environ.copy()
    env["ESG_RESEARCH_ROOT"] = str(RESEARCH_ROOT)
    subprocess.run([sys.executable, str(SCRIPT_DIR / script), *args], check=True, env=env)

def preview_workbook(relative_path, sheet=None, n=5):
    from openpyxl import load_workbook
    from IPython.display import display
    import pandas as pd
    path = COMP_ROOT / relative_path
    if not path.is_file():
        print(f"아직 없음: {path}")
        return
    wb = load_workbook(path, read_only=True, data_only=True)
    print({"file": path.name, "sheets": {name: wb[name].max_row - 1 for name in wb.sheetnames}})
    ws = wb[sheet] if sheet else wb.active
    values = ws.iter_rows(values_only=True)
    headers = next(values, [])
    rows = [list(row) for _, row in zip(range(n), values)]
    df = pd.DataFrame(rows, columns=headers)
    for col in df.select_dtypes(include="object"):
        df[col] = df[col].map(lambda x: x[:120] + "…" if isinstance(x, str) and len(x) > 120 else x)
    if SHOW_PRIVATE_ROWS:
        display(df)
    wb.close()

def preview_csv(relative_path, n=5):
    from IPython.display import display
    import pandas as pd
    path = COMP_ROOT / relative_path
    if not path.is_file():
        print(f"아직 없음: {path}")
        return
    df = pd.read_csv(path, encoding="utf-8-sig")
    print({"file": path.name, "rows": len(df), "columns": len(df.columns)})
    sample = df.head(n).copy()
    for col in sample.select_dtypes(include="object"):
        sample[col] = sample[col].map(lambda x: x[:120] + "…" if isinstance(x, str) and len(x) > 120 else x)
    if SHOW_PRIVATE_ROWS:
        display(sample)

## 1. 학과 목록과 수집 시작 주소

- 교수진에게 받은 대학·학과 목록과 직접 확인한 홈페이지 주소를 확인합니다.
- 각 실행 셀은 해당 단계 산출물을 갱신합니다.

In [ ]:
RUN_ENTITY_REGISTRY = False
if RUN_ENTITY_REGISTRY:
    run_step("build_entity_registry.py")
preview_csv("0_crawling/phase1_entity_registry/entity_registry_kr.csv")
preview_csv("0_crawling/phase1_entity_registry/entity_registry_jp.csv")

In [ ]:
RUN_SEED_REGISTRY = False
if RUN_SEED_REGISTRY:
    run_step("build_seed_registry.py")
preview_workbook("0_crawling/phase2_seed_registry/seed_registry.xlsx")

## 2. URL 수집과 본문 추출

- URL 수집·본문 재수집은 실제 대학 사이트에 접속합니다. `ALLOW_NETWORK`를 확인한 뒤 해당 단계만 실행합니다.
- URL 중복과 본문 추출 결과를 다음 셀에서 확인합니다.

In [ ]:
RUN_URL_CRAWL = False
if RUN_URL_CRAWL:
    run_step("run_full_crawl.py", network=True)
preview_workbook("0_crawling/phase4_url_inventory/url_inventory_full.xlsx", sheet="url_inventory")

In [ ]:
RUN_FINALIZE_INVENTORY = False
if RUN_FINALIZE_INVENTORY:
    run_step("finalize_inventory.py")
preview_csv("0_crawling/phase4_url_inventory/url_inventory_full.csv")
preview_csv("0_crawling/phase4_url_inventory/url_inventory_full_duplicates.csv")

In [ ]:
RUN_SNIPPETS = False
if RUN_SNIPPETS:
    run_step("extract_text.py", "snippets", network=True)
preview_workbook("0_crawling/phase4_url_inventory/url_inventory_full_trafilatura.xlsx")

In [ ]:
RUN_FULL_TEXT = False
if RUN_FULL_TEXT:
    run_step("extract_text.py", "full-text", network=True)
preview_workbook("0_crawling/phase4_url_inventory/url_inventory_full_text.xlsx")

## 3. 페이지 유형 분류와 검증

- GT 풀 생성에는 기존 수동 라벨 자료가 필요합니다. 일부 입력은 비공개 `old/old_v2` 경로에 있습니다.
- GT 풀을 확인하고 분류 결과를 점검합니다.
- LOO 검증과 전체 분류는 API를 호출하므로 `ALLOW_API`를 켠 경우에만 실행합니다.


In [ ]:
RUN_GT_POOL = False
if RUN_GT_POOL:
    run_step("build_gt_pool.py")
preview_workbook("0_crawling/phase5_page_type/gt_pool_v3.xlsx")

In [ ]:
RUN_LOO_VALIDATION = False
if RUN_LOO_VALIDATION:
    run_step("gt364_leave_one_out_validation.py", api=True)
loo_path = CRAWLING_DATA / "phase5_page_type" / "gt364_loo_results.json"
if loo_path.is_file():
    import json
    results = json.loads(loo_path.read_text())
    print({"validation_file": loo_path.name, "result_type": type(results).__name__, "result_count": len(results)})
else:
    print(f"아직 없음: {loo_path}")

In [ ]:
RUN_PAGE_CLASSIFICATION = False
if RUN_PAGE_CLASSIFICATION:
    run_step("classify_dynamic_fewshot_full.py", api=True)
preview_workbook("0_crawling/phase5_page_type/page_type_classified_v3.xlsx")

## 4. 수동 검토

- 검토표를 만든 뒤 사람이 페이지를 열어 유형을 확정합니다.
- 검토표의 판정값을 확인하기 전에는 반영 셀을 실행하지 않습니다.

In [ ]:
RUN_REVIEW_PACKAGE = False
if RUN_REVIEW_PACKAGE:
    run_step("manual_review.py", "prepare")
preview_workbook("0_crawling/phase5_page_type/저신뢰플래그_수동검토_v3.xlsx")

In [ ]:
RUN_APPLY_REVIEW = False
if RUN_APPLY_REVIEW:
    run_step("manual_review.py", "apply", manual_apply=True)
preview_workbook("0_crawling/phase5_page_type/page_type_classified_v3.xlsx")

## 5. 최종 데이터셋

- 수동 검토 결과를 확인한 뒤 생성합니다. 기존 최종 파일을 교체하려면 별도로 허용해야 합니다.
- 분석 입력은 XLSX의 `esg_final_dataset` 시트입니다. 생성 결과의 행 수와 페이지 유형 분포를 확인합니다.


In [ ]:
RUN_FINAL_DATASET = False
final_dir = CRAWLING_DATA / "phase5_page_type"
final_path = final_dir / "esg_final_dataset_v3.xlsx"
final_csv_path = final_dir / "esg_final_dataset_v3.csv"
if RUN_FINAL_DATASET:
    if not CONFIRMED_MANUAL_REVIEW:
        raise RuntimeError("수동 검토 반영 여부를 확인한 뒤 CONFIRMED_MANUAL_REVIEW=True로 설정하세요.")
    if (final_path.exists() or final_csv_path.exists()) and not ALLOW_REPLACE_FINAL_DATASET:
        raise RuntimeError("기존 최종 파일이 있습니다. 교체를 확인한 뒤 ALLOW_REPLACE_FINAL_DATASET=True로 설정하세요.")
    run_step("build_final_dataset.py")
if final_path.is_file():
    import pandas as pd
    from IPython.display import display
    final_df = pd.read_excel(final_path, sheet_name="esg_final_dataset")
    print({"documents": len(final_df), "columns": len(final_df.columns)})
    type_col = next((c for c in ("page_type", "final_page_type", "page_type_final") if c in final_df), None)
    if type_col:
        display(final_df[type_col].fillna("missing").value_counts().rename_axis("page_type").reset_index(name="documents"))
    if SHOW_PRIVATE_ROWS:
        sample = final_df.head(5).copy()
        for col in sample.select_dtypes(include="object"):
            sample[col] = sample[col].map(lambda x: x[:120] + "…" if isinstance(x, str) and len(x) > 120 else x)
        display(sample)
else:
    print(f"아직 없음: {final_path}")
